# 09 - Synthetic control

Sometimes exactly one unit is treated: one region adopts a policy, one store
runs the pilot, one country signs the treaty. There is no control group, only a
pool of untreated units none of which resembles the treated one closely enough
to serve as its counterfactual.

Synthetic control builds the comparison instead of finding it — a weighted
average of untreated units chosen to track the treated unit before treatment,
then extrapolated forward as the counterfactual.

## Causal question

One region adopted a policy at a known date. What would have happened to its
outcome had it not adopted the policy, and how large is the gap?

## Data and design

- **Unit of analysis:** one region observed over time. The panel is balanced.
- **Treatment:** `treatment`, switching on for a single unit from a known period.
- **Outcome:** `outcome`, observed for every unit in every period.
- **Donor pool:** the 29 untreated regions, from which the synthetic control is
  built.
- **Pre-treatment window:** periods 0-7. The post-treatment window is 8-15.

The generator applies a known effect, so the estimate can be compared against
the truth.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

import numpy as np
import pandas as pd

from causal_inference_lab.data_generators import make_synthetic_control_data
from causal_inference_lab.synthetic_control import fit_synthetic_control

dataset = make_synthetic_control_data(
    n_units=30,
    n_periods=16,
    pre_periods=8,
    treated_unit=0,
    effect=3.0,
    seed=101,
)
data = dataset.data

print(f"units:          {data['unit'].nunique()}  (1 treated, {data['unit'].nunique() - 1} donors)")
print(f"periods:        {data['time'].nunique()}  (0-7 pre, 8-15 post)")
print(f"true effect:    {dataset.true_ate:.3f}")
print()
print("outcome by period, treated unit vs donor average:")
summary = (
    data.assign(group=np.where(data["unit"] == 0, "treated", "donors"))
    .groupby(["time", "group"])["outcome"]
    .mean()
    .unstack()
    .round(2)
)
print(summary.to_string())

units:          30  (1 treated, 29 donors)
periods:        16  (0-7 pre, 8-15 post)
true effect:    3.000

outcome by period, treated unit vs donor average:
group  donors  treated
time                  
0        2.97     0.90
1        3.27     1.31
2        3.68     1.83
3        3.88     3.20
4        3.97     2.58
5        4.39     4.51
6        4.45     3.90
7        4.81     4.25
8        4.75     7.81
9        5.55     7.73
10       5.74     6.58
11       5.99     9.92
12       5.97     9.38
13       6.15     9.68
14       6.72     8.11
15       7.10     7.48


**Interpretation.** The treated unit does not track the donor average even
before treatment — it sits persistently higher. That is precisely why a simple
before-and-after comparison against the donor mean would be wrong, and why the
method exists: we need a weighted combination of donors that matches this
particular unit, not the average one.

## Estimand

The **effect on the treated unit** in the post-treatment periods: the difference
between its observed outcome and the outcome it would have had absent
treatment.

This is not an ATE, and not even an ATT in the usual sense. There is one treated
unit, so it is an effect for that unit alone. Nothing here licenses a statement
about regions in general.

## Identification assumptions

1. **A convex combination of donors can reproduce the treated unit's
   pre-treatment path.** If no weighting fits, the counterfactual is
   extrapolation rather than interpolation.
2. **The relationship is stable.** Whatever made the weighted donors track the
   treated unit before treatment continues to hold after it.
3. **No treatment spillover.** Donors are unaffected by the treated unit's
   policy. If the policy diverts activity from neighbours, their outcomes fall,
   the synthetic control falls with them, and the effect is overstated.
4. **No other shock coinciding with treatment** that hits the treated unit and
   not the donors.

Assumption 2 is the load-bearing one and is untestable: it is a claim about a
period in which we can never observe the counterfactual.

## Estimation

The weights are chosen to minimise pre-treatment squared error, subject to being
non-negative and summing to one. Those constraints are what keep the
counterfactual inside the range of observed donor behaviour.

In [2]:
result = fit_synthetic_control(data, treated_unit=0, pre_period_end=7)

print(f"estimated effect:    {result.estimated_effect:.3f}")
print(f"true effect:         {dataset.true_ate:.3f}")
print(f"pre-treatment RMSE:  {result.pre_treatment_rmse:.3f}")

contributing = result.weights.loc[result.weights["weight"] > 0.01].sort_values(
    "weight", ascending=False
)
print(f"\ndonors with meaningful weight: {len(contributing)} of {len(result.weights)}")
print(contributing.to_string(index=False, float_format=lambda v: f"{v:.3f}"))

estimated effect:    3.679
true effect:         3.000
pre-treatment RMSE:  0.377

donors with meaningful weight: 4 of 29
 unit  weight
    1   0.513
   14   0.312
   21   0.128
   23   0.047


**Interpretation.** Two things stand out.

The solution is **sparse**: 4 donors out of 29 carry essentially all the weight,
with one contributing over half. That is characteristic of the simplex
constraint, and it is a feature — the counterfactual is interpretable, traceable
to a handful of named units rather than a diffuse average.

The estimate is **3.68 against a true effect of 3.00**, an overshoot of roughly
23%, despite a pre-treatment RMSE of 0.38 that looks like a good fit. Hold that
thought; the diagnostics below explain it.

## Diagnostics

The pre-treatment fit is the primary diagnostic: a synthetic control that cannot
reproduce the treated unit before treatment has no claim to reproducing it
after. But fit alone is not enough, so we also look at how the estimated gap
behaves period by period.

In [3]:
print(result.effects.to_string(index=False, float_format=lambda v: f"{v:.3f}"))

per_period = result.effects["effect"]
print(f"\nmean effect:   {per_period.mean():.3f}")
print(f"spread:        {per_period.min():.3f} to {per_period.max():.3f}")
print(f"sd:            {per_period.std():.3f}")
print(f"pre-treatment RMSE for reference: {result.pre_treatment_rmse:.3f}")

 time  treated_outcome  synthetic_outcome  effect
    8            7.810              3.391   4.420
    9            7.732              4.396   3.336
   10            6.581              3.917   2.664
   11            9.921              4.850   5.071
   12            9.380              5.022   4.359
   13            9.678              4.468   5.211
   14            8.112              4.970   3.142
   15            7.482              6.253   1.229

mean effect:   3.679
spread:        1.229 to 5.211
sd:            1.349
pre-treatment RMSE for reference: 0.377


**Interpretation.** The per-period effects swing from 1.23 to 5.21 around a mean
of 3.68 — a spread far wider than the pre-treatment RMSE of 0.38. With a single
treated unit there is no averaging across units to damp period-specific noise,
so each period's gap carries the full weight of whatever idiosyncratic shock hit
that region that period.

This is why the point estimate overshoots the truth by 23% while still being a
reasonable answer. The method has correctly detected a substantial positive
effect; it has not pinned down its size to two significant figures, and
reporting 3.68 without this spread would imply a precision the design does not
have.

## Uncertainty

Standard errors are not available here in the usual sense — there is one treated
unit, so there is no sampling distribution to appeal to. The standard approach
is a **placebo permutation test**: pretend in turn that each donor was the
treated unit, fit a synthetic control for it, and compare the resulting effects
against the real one. If the treated unit's effect is unremarkable within that
distribution, it is not evidence of anything.

In [4]:
donors = [u for u in sorted(data["unit"].unique()) if u != 0]
donor_only = data.loc[data["unit"] != 0]

placebo = []
for unit in donors:
    fit = fit_synthetic_control(donor_only, treated_unit=unit, pre_period_end=7)
    placebo.append({"unit": unit, "effect": fit.estimated_effect, "pre_rmse": fit.pre_treatment_rmse})

placebo = pd.DataFrame(placebo)
observed = result.estimated_effect
at_least_as_large = int((placebo["effect"].abs() >= abs(observed)).sum())
p_value = (1 + at_least_as_large) / (1 + len(placebo))

print(f"placebo fits:            {len(placebo)}")
print(f"placebo effects:         mean {placebo['effect'].mean():.3f}, sd {placebo['effect'].std():.3f}")
print(f"placebo range:           [{placebo['effect'].min():.3f}, {placebo['effect'].max():.3f}]")
print(f"observed effect:         {observed:.3f}")
print(f"placebos at least as extreme: {at_least_as_large}")
print(f"permutation p-value:     {p_value:.3f}")

placebo fits:            29
placebo effects:         mean -0.012, sd 0.446
placebo range:           [-1.206, 0.734]
observed effect:         3.679
placebos at least as extreme: 0
permutation p-value:     0.033


**Interpretation.** The placebo effects centre on zero (mean −0.01) with a
standard deviation of 0.45, and none of the 29 reaches the observed 3.68 — the
largest placebo in absolute value is about 1.21. The permutation p-value is
0.033, which is simply 1/30: the treated unit ranks first out of thirty, and
that is the smallest p-value this many donors can produce.

So the effect is real, and the placebo distribution also tells us what "noise"
looks like for this panel: roughly ±0.5. The observed effect sits far outside
it. Note what this test does *not* say — it establishes that an effect exists,
not that 3.68 is its correct magnitude.

A placebo fit is only informative if it was a *good* fit. A donor the method
could not track before treatment will produce a large post-treatment gap for
reasons that have nothing to do with any policy.

In [5]:
threshold = 2 * result.pre_treatment_rmse
well_fitting = placebo.loc[placebo["pre_rmse"] <= threshold]

print(f"treated unit pre-RMSE:      {result.pre_treatment_rmse:.3f}")
print(f"keeping placebos under:     {threshold:.3f}")
print(f"well-fitting placebos:      {len(well_fitting)} of {len(placebo)}")
print(f"of those, at least as extreme as observed: "
      f"{int((well_fitting['effect'].abs() >= abs(observed)).sum())}")

print("\nlargest placebo effects:")
print(
    placebo.reindex(placebo["effect"].abs().sort_values(ascending=False).index)
    .head(5)
    .to_string(index=False, float_format=lambda v: f"{v:.3f}")
)

treated unit pre-RMSE:      0.377
keeping placebos under:     0.754
well-fitting placebos:      24 of 29
of those, at least as extreme as observed: 0

largest placebo effects:
 unit  effect  pre_rmse
    1  -1.206     0.954
   24  -0.785     0.762
   28   0.734     1.014
   17   0.660     0.727
   15  -0.642     0.525


**Interpretation.** Restricting to the 24 placebos that fit at least half as
well as the treated unit does not change the conclusion: none approaches the
observed effect. The inference is not being carried by badly fitted donors,
which is the usual way a placebo test flatters a result.

## Limitations

- **One treated unit.** Every statement here concerns that unit. There is no
  population average effect to report, and no basis for generalising.
- **Inference is permutation-based and coarse.** With 29 donors the smallest
  achievable p-value is 1/30 ≈ 0.033. A smaller donor pool would make even a
  large effect unable to reach conventional significance.
- **The magnitude is imprecise.** The estimate overshoots the known effect by
  23%, and per-period gaps range from 1.2 to 5.2. Detecting an effect and
  measuring it are different achievements.
- **Good pre-treatment fit is necessary, not sufficient.** RMSE 0.38 did not
  prevent the overshoot. Fit measures the past; the assumption concerns the
  future.
- **Spillover would go undetected.** If the policy pulled activity from the
  donor regions, their outcomes fall, the synthetic control falls, and the
  effect inflates — with the pre-treatment fit still looking perfect.
- **Synthetic data.** Donor units here are well-behaved. Real donor pools
  contain units subject to their own policy changes during the window.